# Gradient--Causal Gap: Reproducibility Notebook

This notebook reproduces the experiments, statistical analyses, and figures reported in the paper. The sequence data are generated procedurally; no external dataset is required.

**Important implementation note.** Attention indices are scored using pre-$W_O$ $W_V$ gradient slices, while the reported attention interventions perturb contiguous post-$W_O$ output blocks. Accordingly, these interventions should be interpreted as indexed attention-output perturbations, not exact single-head ablations.


In [ ]:
# %% ── CELL 1: Imports & Config ────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy import stats
from collections import Counter
import pickle, os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)

# Tokens
PAD, START, SEP, END = 100, 101, 102, 103

config = {
    "d_model": 128, "n_heads": 4, "d_ff": 512, "n_layers": 4,
    "vocab_size": 104, "batch_size": 64, "lr": 1e-3,
    "max_train_steps": 15000, "target_train_acc": 0.90,
    "eval_every": 500, "ood_min_acc": 0.20, "ood_max_acc": 0.75,
}

SEEDS      = [42, 123, 456, 789, 1010, 2020, 3030, 4040, 5050, 6060]
THRESHOLDS = [4, 6, 8]   # for sensitivity analysis
TRAIN_MIN, TRAIN_MAX = 3, 7
N_COMP = 20               # 4 layers × (4 heads + 1 MLP)


## Model and helper functions


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads    = n_heads
        self.d_head     = d_model // n_heads
        self.W_q        = nn.Linear(d_model, d_model)
        self.W_k        = nn.Linear(d_model, d_model)
        self.W_v        = nn.Linear(d_model, d_model)
        self.W_o        = nn.Linear(d_model, d_model)
        self.head_outputs = []

    def forward(self, x, mask=None):
        B, T, D = x.shape
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = (Q @ K.transpose(-2, -1)) / (self.d_head ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn      = F.softmax(scores, dim=-1)
        head_out  = attn @ V
        self.head_outputs = [head_out[:, i] for i in range(self.n_heads)]
        return self.W_o(head_out.transpose(1, 2).contiguous().view(B, T, D))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.mlp  = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        return x + self.mlp(self.ln2(x))


class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed  = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([
            TransformerBlock(config["d_model"], config["n_heads"], config["d_ff"])
            for _ in range(config["n_layers"])
        ])
        self.ln_f = nn.LayerNorm(config["d_model"])
        self.out  = nn.Linear(config["d_model"], config["vocab_size"])

    def forward(self, x, mask=None):
        x = self.embed(x)
        for block in self.blocks:
            x = block(x, mask)
        return self.out(self.ln_f(x))

    def get_mask(self, seq_len):
        return torch.tril(
            torch.ones(seq_len, seq_len, device=device)
        ).unsqueeze(0).unsqueeze(0)

 # %% ── CELL 3: Helper Functions ────────────────────────────────

def make_batch(batch_size, min_len, max_len, task):
    inputs, targets = [], []
    for _ in range(batch_size):
        length = np.random.randint(min_len, max_len + 1)
        nums   = np.random.randint(1, 100, size=length).tolist()
        inputs.append([START] + nums + [SEP])
        targets.append((nums[::-1] if task == "reverse" else sorted(nums)) + [END])
    max_inp = max(len(x) for x in inputs)
    max_tgt = max(len(x) for x in targets)
    inputs  = [x + [PAD] * (max_inp - len(x)) for x in inputs]
    targets = [x + [PAD] * (max_tgt - len(x)) for x in targets]
    return torch.tensor(inputs, device=device), torch.tensor(targets, device=device)


def compute_accuracy(model, min_len, max_len, task, n_samples=100):
    model.eval()
    correct = 0
    with torch.no_grad():
        for _ in range(n_samples):
            inp, tgt  = make_batch(1, min_len, max_len, task)
            generated = inp.clone()
            for _ in range(tgt.size(1)):
                logits    = model(generated, model.get_mask(generated.size(1)))
                generated = torch.cat(
                    [generated, logits[:, -1].argmax(-1, keepdim=True)], dim=1
                )
            if (generated[0, inp.size(1):inp.size(1) + tgt.size(1)].tolist()
                    == tgt[0].tolist()):
                correct += 1
    model.train()
    return correct / n_samples


def train_model(model, optimizer, task):
    model.train()
    for step in range(config["max_train_steps"]):
        inp, tgt = make_batch(config["batch_size"], TRAIN_MIN, TRAIN_MAX, task)
        dec_inp  = torch.cat([inp, tgt[:, :-1]], dim=1)
        logits   = model(dec_inp, model.get_mask(dec_inp.size(1)))
        loss     = F.cross_entropy(
            logits[:, inp.size(1)-1:inp.size(1)-1+tgt.size(1)]
                  .reshape(-1, config["vocab_size"]),
            tgt.reshape(-1), ignore_index=PAD,
        )
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        if (step + 1) % config["eval_every"] == 0:
            if compute_accuracy(model, TRAIN_MIN, TRAIN_MAX, task) >= config["target_train_acc"]:
                return step + 1
    return config["max_train_steps"]


def find_ood_length(model, task):
    for test_len in [8, 9, 10, 11]:
        acc = compute_accuracy(model, test_len, test_len, task)
        if config["ood_min_acc"] <= acc <= config["ood_max_acc"]:
            return test_len, acc
    return None, None


def measure_gradients(model, optimizer, test_len, task):
    """Per-head W_V slice gradient norm, averaged over 50 batches."""
    grads = {f"L{l}_H{h}": [] for l in range(4) for h in range(4)}
    grads.update({f"L{l}_MLP": [] for l in range(4)})
    model.train()
    for _ in range(50):
        inp, tgt = make_batch(config["batch_size"], test_len, test_len, task)
        dec_inp  = torch.cat([inp, tgt[:, :-1]], dim=1)
        logits   = model(dec_inp, model.get_mask(dec_inp.size(1)))
        loss     = F.cross_entropy(
            logits[:, inp.size(1)-1:inp.size(1)-1+tgt.size(1)]
                  .reshape(-1, config["vocab_size"]),
            tgt.reshape(-1), ignore_index=PAD,
        )
        optimizer.zero_grad(); loss.backward()
        for l, block in enumerate(model.blocks):
            d_head = config["d_model"] // config["n_heads"]
            for h in range(4):
                grads[f"L{l}_H{h}"].append(
                    block.attn.W_v.weight.grad[h*d_head:(h+1)*d_head, :].norm().item()
                )
            grads[f"L{l}_MLP"].append(block.mlp[2].weight.grad.norm().item())
    return {k: np.mean(v) for k, v in grads.items()}


def _run_ablated_forward(model, inp, tgt, mean_ablations=None, zero_ablations=None):
    """
    Single auto-regressive generation with component interventions.
    mean_ablations : dict {name: tensor}  — replace output with stored mean
    zero_ablations : set of names         — replace output with zeros
    Returns True if generation matches target.
    """
    mean_ablations = mean_ablations or {}
    zero_ablations = zero_ablations or set()
    generated = inp.clone()
    for _ in range(tgt.size(1)):
        x    = model.embed(generated)
        mask = model.get_mask(generated.size(1))
        for l, block in enumerate(model.blocks):
            attn_out = block.attn(block.ln1(x), mask)
            B, T, D  = x.shape
            d_head   = D // 4
            for h in range(4):
                name = f"L{l}_H{h}"
                if name in mean_ablations:
                    attn_out = attn_out.view(B, T, 4, d_head)
                    attn_out[:, :, h, :] = mean_ablations[name]
                    attn_out = attn_out.view(B, T, D)
                elif name in zero_ablations:
                    attn_out = attn_out.view(B, T, 4, d_head)
                    attn_out[:, :, h, :] = torch.zeros(d_head, device=device)
                    attn_out = attn_out.view(B, T, D)
            x       = x + attn_out
            mlp_out = block.mlp(block.ln2(x))
            name    = f"L{l}_MLP"
            if name in mean_ablations:
                mlp_out = mean_ablations[name].unsqueeze(0).unsqueeze(0).expand_as(mlp_out)
            elif name in zero_ablations:
                mlp_out = torch.zeros_like(mlp_out)
            x = x + mlp_out
        logits    = model.out(model.ln_f(x))
        generated = torch.cat([generated, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
    return generated[0, inp.size(1):inp.size(1) + tgt.size(1)].tolist() == tgt[0].tolist()


def _ablate_acc(model, name, test_len, task, means, baseline="mean", n=100, batch_size=20): # computes both mean and zero
    model.eval()
    correct  = 0
    n_iter   = max(1, n // batch_size)
    with torch.no_grad():
        for _ in range(n_iter):
            inp, tgt  = make_batch(batch_size, test_len, test_len, task)
            generated = inp.clone()
            for _ in range(tgt.size(1)):
                x    = model.embed(generated)
                mask = model.get_mask(generated.size(1))
                for l, block in enumerate(model.blocks):
                    attn_out = block.attn(block.ln1(x), mask)
                    B, T, D  = x.shape; d_head = D // 4
                    for h in range(4):
                        cname = f"L{l}_H{h}"
                        if cname == name:
                            attn_out = attn_out.view(B, T, 4, d_head)
                            if baseline == "mean":
                                attn_out[:, :, h, :] = means[name]
                            else:
                                attn_out[:, :, h, :] = torch.zeros(d_head, device=device)
                            attn_out = attn_out.view(B, T, D)
                    x       = x + attn_out
                    mlp_out = block.mlp(block.ln2(x))
                    if f"L{l}_MLP" == name:
                        mlp_out = (means[name].unsqueeze(0).unsqueeze(0).expand_as(mlp_out)
                                   if baseline == "mean" else torch.zeros_like(mlp_out))
                    x = x + mlp_out
                logits    = model.out(model.ln_f(x))
                generated = torch.cat([generated, logits[:, -1].argmax(-1, keepdim=True)], dim=1)
            for b in range(batch_size):
                if (generated[b, inp.size(1):inp.size(1)+tgt.size(1)].tolist()
                        == tgt[b].tolist()):
                    correct += 1
    return correct / (n_iter * batch_size)


def measure_causal_both_baselines(model, baseline_acc, test_len, task):
    """
    Returns effects_mean, effects_zero, means.
    Each effects dict maps component name → drop in accuracy when ablated.
    """
    # Collect mean activations
    means = {f"L{l}_H{h}": [] for l in range(4) for h in range(4)}
    means.update({f"L{l}_MLP": [] for l in range(4)})
    model.eval()
    with torch.no_grad():
        for _ in range(50):
            inp, tgt = make_batch(config["batch_size"], test_len, test_len, task)
            dec_inp  = torch.cat([inp, tgt[:, :-1]], dim=1)
            x = model.embed(dec_inp)
            for l, block in enumerate(model.blocks):
                x = x + block.attn(block.ln1(x), model.get_mask(dec_inp.size(1)))
                for h, ho in enumerate(block.attn.head_outputs):
                    means[f"L{l}_H{h}"].append(ho.mean(dim=(0, 1)).cpu())
                mlp_out = block.mlp(block.ln2(x))
                means[f"L{l}_MLP"].append(mlp_out.mean(dim=(0, 1)).cpu())
                x = x + mlp_out
    means = {k: torch.stack(v).mean(0).to(device) for k, v in means.items()}

    effects_mean, effects_zero = {}, {}
    for l in range(4):
        for h in range(4):
            name = f"L{l}_H{h}"
            effects_mean[name] = baseline_acc - _ablate_acc(model, name, test_len, task, means, "mean")
            effects_zero[name] = baseline_acc - _ablate_acc(model, name, test_len, task, means, "zero")
        name = f"L{l}_MLP"
        effects_mean[name] = baseline_acc - _ablate_acc(model, name, test_len, task, means, "mean")
        effects_zero[name] = baseline_acc - _ablate_acc(model, name, test_len, task, means, "zero")

    return effects_mean, effects_zero, means


def classify(g_rank, c_rank, threshold=6):
    diff = g_rank - c_rank
    if diff < -threshold: return "hidden_hero"
    if diff >  threshold: return "gradient_bloat"
    return "aligned"


## Main experiments


In [ ]:

def run_experiment(task):
    print(f"\n{'='*60}\n{task.upper()} TASK\n{'='*60}")
    results, skipped = [], []

    for i, seed in enumerate(SEEDS):
        print(f"\nSeed {seed} ({i+1}/{len(SEEDS)})")
        torch.manual_seed(seed); np.random.seed(seed)
        model     = Transformer().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])

        train_model(model, optimizer, task)
        test_len, ood_acc = find_ood_length(model, task)
        if test_len is None:
            print("  SKIPPED — no valid OOD length found")
            skipped.append(seed); continue
        print(f"  OOD len={test_len}  acc={ood_acc:.0%}")

        grads = measure_gradients(model, optimizer, test_len, task)
        effects_mean, effects_zero, means = measure_causal_both_baselines(
            model, ood_acc, test_len, task
        )

        comps       = sorted(grads.keys())
        G           = np.array([grads[c]         for c in comps])
        C_mean      = np.array([effects_mean[c]  for c in comps])
        C_zero      = np.array([effects_zero[c]  for c in comps])
        G_rank      = stats.rankdata(G)
        C_rank_mean = stats.rankdata(C_mean)
        C_rank_zero = stats.rankdata(C_zero)
        rho_mean, _ = stats.spearmanr(G, C_mean)
        rho_zero, _ = stats.spearmanr(G, C_zero)

        types_mean = {c: classify(G_rank[j], C_rank_mean[j]) for j, c in enumerate(comps)}
        types_zero = {c: classify(G_rank[j], C_rank_zero[j]) for j, c in enumerate(comps)}
        heroes = [c for c in comps if types_mean[c] == "hidden_hero"]
        bloats = [c for c in comps if types_mean[c] == "gradient_bloat"]

        print(f"  ρ(mean)={rho_mean:.3f}  ρ(zero)={rho_zero:.3f}  "
              f"Heroes={len(heroes)}  Bloats={len(bloats)}")

        results.append({
            "seed": seed, "task": task,
            "test_len": test_len, "ood_acc": ood_acc,
            # Keep 'rho' alias for backward compat with plotting code
            "rho": rho_mean, "rho_mean": rho_mean, "rho_zero": rho_zero,
            "grads": grads,
            # Keep 'causal' alias for backward compat
            "causal": effects_mean,
            "effects_mean": effects_mean, "effects_zero": effects_zero,
            # Keep 'C_rank' alias for backward compat
            "C_rank": C_rank_mean,
            "G_rank": G_rank, "C_rank_mean": C_rank_mean, "C_rank_zero": C_rank_zero,
            # Keep 'types' alias for backward compat
            "types": types_mean,
            "types_mean": types_mean, "types_zero": types_zero,
            # Keep heroes/bloats as lists of tuples AND plain lists
            "heroes": heroes, "bloats": bloats,
            "means": means,
        })

    return results, skipped


reverse_results, rev_skipped = run_experiment("reverse")
sort_results,    sort_skipped = run_experiment("sort")

with open(f"{output_dir}/all_results.pkl", "wb") as f:
    pickle.dump({"reverse": reverse_results, "sort": sort_results}, f)
print(f"\nSaved → {output_dir}/all_results.pkl")

## Top-two pruning


In [ ]:
# %% ── CELL 5: 10-Seed Pruning ─────────────────────────────────

def run_pruning_10seed(task, all_results):
    print(f"\n{'='*60}\n10-SEED PRUNING — {task.upper()}\n{'='*60}")
    rows = []

    for r in all_results:
        seed     = r["seed"]
        test_len = r["test_len"]
        print(f"\n  Seed {seed}  OOD len={test_len}")

        torch.manual_seed(seed); np.random.seed(seed)
        model     = Transformer().to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
        train_model(model, optimizer, task)

        baseline = compute_accuracy(model, test_len, test_len, task, n_samples=200)
        means    = r["means"]

        bloat_comps = r["bloats"][:2]
        hero_comps  = r["heroes"][:2]

        def multi_prune_acc(comps_to_prune):
            model.eval()
            correct = 0
            ablations = {c: means[c] for c in comps_to_prune if c in means}
            with torch.no_grad():
                for _ in range(200):
                    inp, tgt = make_batch(1, test_len, test_len, task)
                    if _run_ablated_forward(model, inp, tgt, mean_ablations=ablations):
                        correct += 1
            return correct / 200

        acc_bloats = multi_prune_acc(bloat_comps) if bloat_comps else baseline
        acc_heroes = multi_prune_acc(hero_comps)  if hero_comps  else baseline

        row = {
            "seed": seed, "baseline": baseline,
            "bloat_comps": bloat_comps, "hero_comps": hero_comps,
            "acc_bloats": acc_bloats,   "acc_heroes": acc_heroes,
            "drop_bloats": baseline - acc_bloats,
            "drop_heroes": baseline - acc_heroes,
        }
        print(f"    Baseline={baseline:.1%}  Prune Bloats={acc_bloats:.1%}  "
              f"Prune Heroes={acc_heroes:.1%}")
        rows.append(row)

    drops_b = [r["drop_bloats"] for r in rows]
    drops_h = [r["drop_heroes"] for r in rows]
    print(f"\n  Bloat prune drop: {np.mean(drops_b):.1%} ± {np.std(drops_b):.1%}")
    print(f"  Hero  prune drop: {np.mean(drops_h):.1%} ± {np.std(drops_h):.1%}")
    return rows


rev_pruning  = run_pruning_10seed("reverse", reverse_results)
sort_pruning = run_pruning_10seed("sort",    sort_results)

with open(f"{output_dir}/pruning_10seed.pkl", "wb") as f:
    pickle.dump({"reverse": rev_pruning, "sort": sort_pruning}, f)
print(f"Saved → {output_dir}/pruning_10seed.pkl")


## Threshold sensitivity


In [ ]:
def threshold_sensitivity(task, all_results):
    print(f"\n{'='*60}\nTHRESHOLD SENSITIVITY — {task.upper()}\n{'='*60}")
    # layer_counts[threshold][layer] = {"hero": int, "bloat": int}
    layer_counts = {t: {l: {"hero": 0, "bloat": 0} for l in range(4)} for t in THRESHOLDS}

    for r in all_results:
        comps  = sorted(r["grads"].keys())
        G_rank = r["G_rank"]
        C_rank = r["C_rank_mean"]
        for j, c in enumerate(comps):
            if "MLP" in c:
                continue
            layer = int(c[1])
            for t in THRESHOLDS:
                label = classify(G_rank[j], C_rank[j], threshold=t)
                if label == "hidden_hero":
                    layer_counts[t][layer]["hero"]  += 1
                elif label == "gradient_bloat":
                    layer_counts[t][layer]["bloat"] += 1

    for t in THRESHOLDS:
        print(f"\n  Threshold ±{t}:")
        for l in range(4):
            h = layer_counts[t][l]["hero"]
            b = layer_counts[t][l]["bloat"]
            print(f"    Layer {l}: Heroes={h}  Bloats={b}")

    return layer_counts


rev_thresh  = threshold_sensitivity("reverse", reverse_results)
sort_thresh = threshold_sensitivity("sort",    sort_results)

## Statistical analysis


In [ ]:
# Statistical analysis and reproducibility outputs
import json
import pandas as pd

RESULTS_DIR = "./results"
ALL_RESULTS_PATH = os.path.join(RESULTS_DIR, "all_results.pkl")
PRUNING_PATH = os.path.join(RESULTS_DIR, "pruning_10seed.pkl")

N_BOOTSTRAP = 10_000
BOOTSTRAP_SEED = 1234
rng = np.random.default_rng(BOOTSTRAP_SEED)

def _to_native(x):
    if isinstance(x, dict):
        return {k: _to_native(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_native(v) for v in x]
    if isinstance(x, np.floating):
        return float(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.ndarray):
        return _to_native(x.tolist())
    return x

def bootstrap_mean_ci(values, n_boot=N_BOOTSTRAP, rng=rng, ci=95):
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return {
            "point_estimate": None, "std": None,
            "ci_low": None, "ci_high": None,
            "n_seeds": 0, "n_boot": n_boot,
        }
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = values[idx].mean(axis=1)
    lo, hi = np.percentile(
        boot_means, [(100-ci)/2, 100-(100-ci)/2]
    )
    return {
        "point_estimate": float(values.mean()),
        "std": float(values.std(ddof=1)) if n > 1 else 0.0,
        "ci_low": float(lo),
        "ci_high": float(hi),
        "n_seeds": int(n),
        "n_boot": int(n_boot),
    }

def paired_delta_ci(rev_by_seed, sort_by_seed, seeds,
                    n_boot=N_BOOTSTRAP, rng=rng, ci=95):
    seeds = list(seeds)
    n = len(seeds)
    rev_vals = np.array([rev_by_seed[s] for s in seeds], dtype=float)
    sort_vals = np.array([sort_by_seed[s] for s in seeds], dtype=float)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_delta = rev_vals[idx].mean(axis=1) - sort_vals[idx].mean(axis=1)
    lo, hi = np.percentile(
        boot_delta, [(100-ci)/2, 100-(100-ci)/2]
    )
    return {
        "point_estimate": float(rev_vals.mean() - sort_vals.mean()),
        "ci_low": float(lo),
        "ci_high": float(hi),
        "n_seeds": int(n),
        "n_boot": int(n_boot),
    }

with open(ALL_RESULTS_PATH, "rb") as f:
    all_results = pickle.load(f)

reverse_results = all_results["reverse"]
sort_results = all_results["sort"]

rev_seeds_valid = sorted(r["seed"] for r in reverse_results)
sort_seeds_valid = sorted(r["seed"] for r in sort_results)
all_seeds_seen = sorted(set(rev_seeds_valid) | set(sort_seeds_valid))
common_seeds = sorted(set(rev_seeds_valid) & set(sort_seeds_valid))

ci_rev_mean = bootstrap_mean_ci([r["rho_mean"] for r in reverse_results])
ci_rev_zero = bootstrap_mean_ci([r["rho_zero"] for r in reverse_results])
ci_sort_mean = bootstrap_mean_ci([r["rho_mean"] for r in sort_results])
ci_sort_zero = bootstrap_mean_ci([r["rho_zero"] for r in sort_results])

rev_mean_by_seed = {r["seed"]: r["rho_mean"] for r in reverse_results}
rev_zero_by_seed = {r["seed"]: r["rho_zero"] for r in reverse_results}
sort_mean_by_seed = {r["seed"]: r["rho_mean"] for r in sort_results}
sort_zero_by_seed = {r["seed"]: r["rho_zero"] for r in sort_results}

delta_mean = paired_delta_ci(
    rev_mean_by_seed, sort_mean_by_seed, common_seeds
)
delta_zero = paired_delta_ci(
    rev_zero_by_seed, sort_zero_by_seed, common_seeds
)

pruning_cis = None
if os.path.exists(PRUNING_PATH):
    with open(PRUNING_PATH, "rb") as f:
        pruning_data = pickle.load(f)

    pruning_cis = {}
    for task_label, rows in [
        ("Reversal", pruning_data["reverse"]),
        ("Sorting", pruning_data["sort"]),
    ]:
        ci_hero = bootstrap_mean_ci([r["drop_heroes"] for r in rows])
        ci_bloat = bootstrap_mean_ci([r["drop_bloats"] for r in rows])
        pruning_cis[task_label] = {
            "hero_drop": ci_hero,
            "bloat_drop": ci_bloat,
        }

manifest_rows = []
for task_label, results in [
    ("reverse", reverse_results),
    ("sort", sort_results),
]:
    by_seed = {r["seed"]: r for r in results}
    for seed in SEEDS:
        r = by_seed.get(seed)
        manifest_rows.append({
            "task": task_label,
            "seed": seed,
            "status": "valid" if r is not None else "skipped",
            "ood_length": r["test_len"] if r else None,
            "ood_accuracy": r["ood_acc"] if r else None,
            "rho_mean_ablation": r["rho_mean"] if r else None,
            "rho_zero_ablation": r["rho_zero"] if r else None,
            "n_hidden_heroes": len(r["heroes"]) if r else None,
            "n_gradient_bloats": len(r["bloats"]) if r else None,
        })

manifest_df = pd.DataFrame(manifest_rows)

ood_rows = []
for seed in SEEDS:
    rr = next((r for r in reverse_results if r["seed"] == seed), None)
    sr = next((r for r in sort_results if r["seed"] == seed), None)
    ood_rows.append({
        "Seed": seed,
        "Reversal OOD Len": rr["test_len"] if rr else None,
        "Reversal OOD Acc": rr["ood_acc"] if rr else None,
        "Sorting OOD Len": sr["test_len"] if sr else None,
        "Sorting OOD Acc": sr["ood_acc"] if sr else None,
    })
ood_df = pd.DataFrame(ood_rows)

bootstrap_output = _to_native({
    "config": {
        "n_bootstrap": N_BOOTSTRAP,
        "bootstrap_rng_seed": BOOTSTRAP_SEED,
        "ci_method": "percentile",
        "resampling_unit": "seed",
    },
    "seed_counts": {
        "reversal_valid": len(rev_seeds_valid),
        "sorting_valid": len(sort_seeds_valid),
        "common_to_both": len(common_seeds),
    },
    "rho_cis": {
        "reversal_mean_ablation": ci_rev_mean,
        "reversal_zero_ablation": ci_rev_zero,
        "sorting_mean_ablation": ci_sort_mean,
        "sorting_zero_ablation": ci_sort_zero,
    },
    "delta_rho_reversal_minus_sorting": {
        "mean_ablation": delta_mean,
        "zero_ablation": delta_zero,
    },
    "pruning_cis": pruning_cis,
})

with open(os.path.join(RESULTS_DIR, "bootstrap_statistics.json"), "w") as f:
    json.dump(bootstrap_output, f, indent=2)

manifest_df.to_csv(
    os.path.join(RESULTS_DIR, "reproducibility_manifest.csv"),
    index=False,
)
ood_df.to_csv(
    os.path.join(RESULTS_DIR, "ood_seed_table.csv"),
    index=False,
)

print("Valid Reversal seeds:", rev_seeds_valid)
print("Valid Sorting seeds:", sort_seeds_valid)
print("Common seeds:", common_seeds)
print("\nMean-ablation rho:")
print("  Reversal:", ci_rev_mean)
print("  Sorting: ", ci_sort_mean)
print("\nZero-ablation rho:")
print("  Reversal:", ci_rev_zero)
print("  Sorting: ", ci_sort_zero)
print("\nPaired delta rho:")
print("  Mean:", delta_mean)
print("  Zero:", delta_zero)
if pruning_cis is not None:
    print("\nPruning CIs:")
    print(pruning_cis)


## Final paper figures


In [ ]:
# ============================================================
# NEURIPS 2026 — FINAL FIGURE REGENERATION
# Plotting only. NO training / NO inference.
# Uses saved results from ./results/
# ============================================================

import os
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

RESULTS_DIR = "./results"
ALL_RESULTS_PATH = os.path.join(RESULTS_DIR, "all_results.pkl")
PRUNING_PATH = os.path.join(RESULTS_DIR, "pruning_10seed.pkl")
BOOTSTRAP_PATH = os.path.join(RESULTS_DIR, "bootstrap_statistics.json")

# ------------------------------------------------------------
# 1. Check files
# ------------------------------------------------------------

for path in [ALL_RESULTS_PATH, PRUNING_PATH, BOOTSTRAP_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Upload/copy the authoritative rerun file into ./results/ first."
        )

with open(ALL_RESULTS_PATH, "rb") as f:
    all_results = pickle.load(f)

with open(PRUNING_PATH, "rb") as f:
    pruning = pickle.load(f)

with open(BOOTSTRAP_PATH, "r") as f:
    bootstrap = json.load(f)

reverse_results = all_results["reverse"]
sort_results = all_results["sort"]

rev_pruning = pruning["reverse"]
sort_pruning = pruning["sort"]

print(
    f"Loaded {len(reverse_results)} Reversal seeds "
    f"and {len(sort_results)} Sorting seeds."
)

# ------------------------------------------------------------
# 2. Basic style
# ------------------------------------------------------------

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ============================================================
# FIGURE 1 — MAIN SUMMARY FIGURE
# A: per-seed mean-ablation rho
# B: Sorting layer-wise Hero/Bloat counts
# C: top-two pruning with bootstrap 95% CIs
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(13.2, 3.8),
    gridspec_kw={"width_ratios": [1.35, 1.0, 1.15]}
)

# ------------------------------------------------------------
# Panel A — Per-seed mean-ablation correlations
# ------------------------------------------------------------

ax = axes[0]

rev_by_seed = {r["seed"]: r["rho_mean"] for r in reverse_results}
sort_by_seed = {r["seed"]: r["rho_mean"] for r in sort_results}

all_seeds = sorted(set(rev_by_seed) | set(sort_by_seed))
x = np.arange(len(all_seeds))

rev_y = [rev_by_seed.get(s, np.nan) for s in all_seeds]
sort_y = [sort_by_seed.get(s, np.nan) for s in all_seeds]

ax.scatter(
    x - 0.10, rev_y,
    s=42, marker="o",
    label="Reversal"
)

ax.scatter(
    x + 0.10, sort_y,
    s=42, marker="s",
    label="Sorting"
)

rev_mean = bootstrap["rho_cis"]["reversal_mean_ablation"]["point_estimate"]
sort_mean = bootstrap["rho_cis"]["sorting_mean_ablation"]["point_estimate"]

ax.axhline(
    rev_mean,
    linestyle="--",
    linewidth=1.2,
    alpha=0.75
)

ax.axhline(
    sort_mean,
    linestyle=":",
    linewidth=1.4,
    alpha=0.75
)

ax.axhline(0, linewidth=0.8, alpha=0.5)

ax.set_xticks(x)
ax.set_xticklabels(all_seeds, rotation=45)
ax.set_xlabel("Random seed")
ax.set_ylabel(r"Spearman $\rho$")
ax.set_title("(a) Gradient--causal alignment")
ax.set_ylim(-0.15, 1.0)
ax.legend(frameon=False)

ax.text(
    0.02, 0.95,
    rf"Mean: Rev.={rev_mean:.3f}, Sort.={sort_mean:.3f}",
    transform=ax.transAxes,
    va="top",
    fontsize=8.5
)

# ------------------------------------------------------------
# Panel B — Sorting layer-wise counts, mean ablation, tau=6
# Uses SAVED strict classifier results.
# Attention indices only.
# ------------------------------------------------------------

ax = axes[1]

hero_counts = [0, 0, 0, 0]
bloat_counts = [0, 0, 0, 0]

for r in sort_results:
    for comp, label in r["types_mean"].items():
        if "MLP" in comp:
            continue

        layer = int(comp[1])

        if label == "hidden_hero":
            hero_counts[layer] += 1
        elif label == "gradient_bloat":
            bloat_counts[layer] += 1

expected_heroes = [6, 0, 8, 17]
expected_bloats = [9, 24, 2, 0]

assert hero_counts == expected_heroes, (
    f"Unexpected Sorting Hero counts: {hero_counts}"
)
assert bloat_counts == expected_bloats, (
    f"Unexpected Sorting Bloat counts: {bloat_counts}"
)

layers = np.arange(4)
width = 0.36

bars_h = ax.bar(
    layers - width/2,
    hero_counts,
    width,
    label="Hidden Heroes"
)

bars_b = ax.bar(
    layers + width/2,
    bloat_counts,
    width,
    label="Gradient Bloats"
)

for bars in [bars_h, bars_b]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width()/2,
                height + 0.5,
                f"{int(height)}",
                ha="center",
                va="bottom",
                fontsize=8
            )

ax.set_xticks(layers)
ax.set_xticklabels(["L0", "L1", "L2", "L3"])
ax.set_ylabel("Occurrences across 10 seeds")
ax.set_title(r"(b) Sorting rank disagreements ($\tau=6$)")
ax.set_ylim(0, 28)
ax.legend(frameon=False)

# ------------------------------------------------------------
# Panel C — Pruning effects + authoritative bootstrap CIs
# ------------------------------------------------------------

ax = axes[2]

pruning_cis = bootstrap["pruning_cis"]

entries = [
    ("Rev.\nHeroes",  "Reversal", "hero_drop"),
    ("Rev.\nBloats",  "Reversal", "bloat_drop"),
    ("Sort.\nHeroes", "Sorting",  "hero_drop"),
    ("Sort.\nBloats", "Sorting",  "bloat_drop"),
]

labels = []
means = []
lower_errors = []
upper_errors = []

for label, task, key in entries:
    d = pruning_cis[task][key]

    point = 100 * d["point_estimate"]
    low   = 100 * d["ci_low"]
    high  = 100 * d["ci_high"]

    labels.append(label)
    means.append(point)
    lower_errors.append(point - low)
    upper_errors.append(high - point)

means = np.array(means)
yerr = np.array([lower_errors, upper_errors])

x2 = np.arange(len(labels))

bars = ax.bar(
    x2,
    means,
    yerr=yerr,
    capsize=4,
    width=0.68
)

for bar, value in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 2.0,
        f"{value:.1f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

ax.set_xticks(x2)
ax.set_xticklabels(labels)
ax.set_ylabel("OOD accuracy drop\n(percentage points)")
ax.set_title("(c) Top-two pruning")
ax.set_ylim(0, 58)

# ------------------------------------------------------------
# Finish Figure 1
# ------------------------------------------------------------

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.22)

plt.tight_layout()

main_pdf = os.path.join(RESULTS_DIR, "figure1_main_summary.pdf")
main_png = os.path.join(RESULTS_DIR, "figure1_main_summary.png")

plt.savefig(main_pdf, bbox_inches="tight")
plt.savefig(main_png, bbox_inches="tight")
plt.show()

print("Saved:", main_pdf)
print("Saved:", main_png)


# ============================================================
# FIGURE A1 — THRESHOLD SENSITIVITY
# Sorting attention indices only
# Mean + zero ablation
# ============================================================

def classify_from_ranks(g_rank, c_rank, threshold):
    diff = g_rank - c_rank
    if diff < -threshold:
        return "hidden_hero"
    if diff > threshold:
        return "gradient_bloat"
    return "aligned"


def layer_counts(results, baseline, threshold):
    heroes = [0, 0, 0, 0]
    bloats = [0, 0, 0, 0]

    for r in results:
        comps = sorted(r["grads"].keys())
        g_rank = np.asarray(r["G_rank"])

        if baseline == "mean":
            c_rank = np.asarray(r["C_rank_mean"])
        else:
            c_rank = np.asarray(r["C_rank_zero"])

        for j, comp in enumerate(comps):
            if "MLP" in comp:
                continue

            layer = int(comp[1])
            label = classify_from_ranks(
                g_rank[j],
                c_rank[j],
                threshold
            )

            if label == "hidden_hero":
                heroes[layer] += 1
            elif label == "gradient_bloat":
                bloats[layer] += 1

    return heroes, bloats


thresholds = [4, 6, 8]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))

for ax, baseline, title in [
    (axes[0], "mean", "Mean ablation"),
    (axes[1], "zero", "Zero ablation"),
]:
    x = np.arange(4)
    offsets = [-0.26, 0.0, 0.26]

    for offset, threshold in zip(offsets, thresholds):
        heroes, bloats = layer_counts(
            sort_results,
            baseline,
            threshold
        )

        # Plot signed count:
        # Heroes negative, Bloats positive
        signed = np.array(bloats) - np.array(heroes)

        ax.bar(
            x + offset,
            signed,
            width=0.23,
            label=rf"$\tau={threshold}$"
        )

    ax.axhline(0, linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(["L0", "L1", "L2", "L3"])
    ax.set_xlabel("Layer")
    ax.set_title(title)
    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

axes[0].set_ylabel(
    "Bloat minus Hero occurrences\n"
    "(positive = more Bloats)"
)
axes[1].legend(frameon=False)

plt.tight_layout()

path_pdf = os.path.join(
    RESULTS_DIR,
    "figureA1_threshold_sensitivity.pdf"
)
path_png = os.path.join(
    RESULTS_DIR,
    "figureA1_threshold_sensitivity.png"
)

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, bbox_inches="tight")
plt.show()

print("Saved:", path_pdf)
print("Saved:", path_png)


# ============================================================
# FIGURE A2 — RAW ATTENTION W_V GRADIENT NORM BY LAYER
# ============================================================

def collect_layer_gradient_norms(results):
    layer_values = defaultdict(list)

    for r in results:
        for comp, value in r["grads"].items():
            if "MLP" in comp:
                continue

            layer = int(comp[1])
            layer_values[layer].append(float(value))

    return layer_values


rev_grad = collect_layer_gradient_norms(reverse_results)
sort_grad = collect_layer_gradient_norms(sort_results)

rev_means = np.array([
    np.mean(rev_grad[l]) for l in range(4)
])

sort_means = np.array([
    np.mean(sort_grad[l]) for l in range(4)
])

print("\nRaw attention W_V gradient means:")
print("Reversal:", np.round(rev_means, 3))
print("Sorting: ", np.round(sort_means, 3))

# Validate against authoritative rerun values
assert np.allclose(
    rev_means,
    [0.615, 0.624, 0.234, 0.149],
    atol=0.001
)

assert np.allclose(
    sort_means,
    [0.156, 0.194, 0.103, 0.054],
    atol=0.001
)

fig, ax = plt.subplots(figsize=(5.8, 4.0))

x = np.arange(4)
width = 0.36

ax.bar(
    x - width/2,
    rev_means,
    width,
    label="Reversal"
)

ax.bar(
    x + width/2,
    sort_means,
    width,
    label="Sorting"
)

ax.set_xticks(x)
ax.set_xticklabels(["L0", "L1", "L2", "L3"])
ax.set_xlabel("Layer")
ax.set_ylabel(r"Mean raw $W_V$ gradient norm")
ax.set_title("Attention gradient magnitude by layer")
ax.legend(frameon=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", linestyle="--", alpha=0.22)

plt.tight_layout()

path_pdf = os.path.join(
    RESULTS_DIR,
    "figureA2_layer_gradient_norms.pdf"
)
path_png = os.path.join(
    RESULTS_DIR,
    "figureA2_layer_gradient_norms.png"
)

plt.savefig(path_pdf, bbox_inches="tight")
plt.savefig(path_png, bbox_inches="tight")
plt.show()

print("Saved:", path_pdf)
print("Saved:", path_png)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 65)
print("FIGURE REGENERATION COMPLETE")
print("=" * 65)

print("""
Main paper:
  figure1_main_summary.pdf
  figure1_main_summary.png

Appendix:
  figureA1_threshold_sensitivity.pdf
  figureA1_threshold_sensitivity.png
  figureA2_layer_gradient_norms.pdf
  figureA2_layer_gradient_norms.png
""")
